# AlegroCode — Colab GPU Server

| Ячейка | Что делает | Когда запускать |
|--------|-----------|------------------|
| **1 — INSTALL** | Клонирует репо, задаёт env vars, ставит зависимости | Один раз после запуска среды |
| **2 — START** | Загружает модели, поднимает сервер, открывает ngrok | При каждом запуске или **рестарте** |
| **3 — STOP** | Останавливает сервер (модели остаются в памяти) | Когда нужно остановить |

> **Рестарт без перезагрузки ядра**: повторно запустите ячейку **2 — START** —
> она сама остановит старый сервер и поднимет новый. Модели не перезагружаются.
>
> **Единственное что нужно заполнить**: блок `ЗАПОЛНИТЕ ЭТИ ЗНАЧЕНИЯ` в ячейке 1.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 — INSTALL  (один раз на сессию)                             ║
# ╚══════════════════════════════════════════════════════════════════╝

# ┌─────────────────────────────────────────────────────────────────┐
# │  ЗАПОЛНИТЕ ЭТИ ЗНАЧЕНИЯ ПЕРЕД ЗАПУСКОМ                         │
# └─────────────────────────────────────────────────────────────────┘
REPO_URL         = 'https://github.com/YOUR_USER/building_analyzer.git'
NGROK_AUTHTOKEN  = ''        # токен с dashboard.ngrok.com/get-started/your-authtoken
INPAINT_PROVIDER = 'lama'   # 'lama' или 'sd'
DB_PATH          = '/work/building_analyzer/data/alegrocode.db'
SCRAPER_ENABLED  = 'false'  # 'true' / 'false'
# ──────────────────────────────────────────────────────────────────

import os, sys, subprocess, textwrap
from pathlib import Path

# ── 1. Клонируем репозиторий (если ещё не скачан) ────────────────────────
repo_root = Path('/work/building_analyzer')
if not repo_root.exists():
    print('Клонируем репозиторий...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(repo_root)],
        check=True
    )
    print('Репозиторий склонирован в', repo_root)
else:
    print('Репозиторий уже есть:', repo_root)

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── 2. Выставляем переменные окружения ───────────────────────────────────
if NGROK_AUTHTOKEN:
    os.environ['ALEGRO_NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
os.environ['ALEGRO_INPAINT_PROVIDER'] = INPAINT_PROVIDER
os.environ['ALEGRO_DB_PATH']           = DB_PATH
os.environ['ALEGRO_SCRAPER_ENABLED']   = SCRAPER_ENABLED
os.environ.pop('PIP_CONSTRAINT', None)
print('Env vars: OK')

# ── 3. Фиксируем packaging ДО любых других установок ─────────────────────
# nvidia-dali и Grounding DINO требуют packaging<=24.2;
# pip upgrade его обновляет до 26.x и ломает _TrimmedRelease импорт.
%pip install -q --no-deps "packaging<=24.2"

# ── 4. Pip tools (без -U чтобы не поднять packaging) ─────────────────────
%pip install -q "pip>=24,<26.1" setuptools wheel

# ── 5. Constraints-файл ───────────────────────────────────────────────────
(repo_root / 'constraints-colab.txt').write_text(textwrap.dedent("""
    numpy>=1.26,<2.0
    opencv-python-headless==4.10.0.84
    matplotlib>=3.8,<3.11
    packaging<=24.2
    scipy>=1.12,<2.0
    transformers>=4.44,<4.46
    accelerate>=0.33,<0.35
    tokenizers>=0.19,<0.20
    huggingface-hub>=0.24,<1.0
    tqdm>=4.66,<5
""").strip() + '\n', encoding='utf-8')

# ── 6. Server-critical пакеты (устанавливаем первыми — независимо) ───────
# Если ML-пакеты ниже упадут, сервер всё равно запустится.
%pip install -q --prefer-binary -c constraints-colab.txt \
    "fastapi==0.115.*" "uvicorn[standard]==0.34.*" \
    python-multipart "sqlalchemy>=2,<3" aiosqlite \
    "pydantic>=2.7" "pydantic-settings>=2.3" \
    httpx pyngrok nest_asyncio

# ── 7. HF/CV стек (пинается первым, чтобы не было resolution-too-deep) ────
%pip install -q --prefer-binary -c constraints-colab.txt \
    "numpy>=1.26,<2.0" "opencv-python-headless==4.10.0.84" "matplotlib>=3.8,<3.11" \
    "transformers>=4.44,<4.46" "accelerate>=0.33,<0.35" \
    "tokenizers>=0.19,<0.20" "huggingface-hub>=0.24,<1.0"

# ── 8. scipy отдельно с --only-binary (scipy>=1.12 имеет wheel для Py3.12) ─
# Без этого pip пытается собрать из исходников — нет OpenBLAS → ошибка.
%pip install -q --only-binary :all: -c constraints-colab.txt "scipy>=1.12,<2.0"

# ── 9. scikit-image после scipy (повторно не устанавливаем scipy) ─────────
%pip install -q --prefer-binary "scikit-image>=0.22"

# ── 10. Остальные зависимости из requirements.txt ─────────────────────────
# scikit-image и тяжёлые пакеты HF уже установлены выше — фильтруем их.
src_req = repo_root / 'backend' / 'requirements.txt'
flt_req = repo_root / 'backend' / 'requirements.colab.filtered.txt'
skip = (
    'torch', 'torchvision', 'transformers', 'accelerate', 'tokenizers',
    'huggingface-hub', 'numpy', 'opencv-python-headless', 'matplotlib',
    'scikit-image',  # установлен в шаге 9
)
lines = []
for line in src_req.read_text(encoding='utf-8').splitlines():
    s = line.strip()
    if not s or s.startswith('#') or not s.startswith(skip):
        lines.append(line)
flt_req.write_text('\n'.join(lines) + '\n', encoding='utf-8')

%pip install -q --prefer-binary -r backend/requirements.colab.filtered.txt -c constraints-colab.txt

# ── 11. SAM2 ──────────────────────────────────────────────────────────────
%pip install -q --no-build-isolation --no-deps \
    git+https://github.com/facebookresearch/sam2.git

import nest_asyncio
nest_asyncio.apply()

import numpy, cv2, sqlalchemy, fastapi
print(f"numpy {numpy.__version__} | cv2 {cv2.__version__} | "
      f"fastapi {fastapi.__version__} | sqlalchemy {sqlalchemy.__version__}")
print("nest_asyncio: OK")
print("\n✅  Установка завершена — запустите ячейку 2 (START)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 — START  (запуск или рестарт сервера)                       ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Запускайте повторно — ячейка сама:
#   • остановит старый сервер (если был запущен)
#   • освободит порт 8000
#   • перезапустит ngrok
#   • НЕ перезагружает модели (они живут в _analyzer)

import asyncio, subprocess, time, os
import nest_asyncio
nest_asyncio.apply()

ngrok_token = os.environ.get('ALEGRO_NGROK_AUTHTOKEN', '')
if not ngrok_token:
    print('⚠️  ALEGRO_NGROK_AUTHTOKEN не задан — ngrok не подключится.')

# ── Остановить старый сервер если есть ───────────────────────────────────
_server = globals().get('_server')
_server_task = globals().get('_server_task')
if _server is not None:
    print('Останавливаем предыдущий сервер...')
    _server.should_exit = True
    try:
        await asyncio.wait_for(_server_task, timeout=8)
    except Exception:
        pass
    print('Предыдущий сервер остановлен')

# Освобождаем порт на случай, если процесс завис
subprocess.run(
    "fuser -k 8000/tcp 2>/dev/null || lsof -ti:8000 | xargs kill -9 2>/dev/null || true",
    shell=True, capture_output=True
)
time.sleep(0.5)

# ── Отключить старый ngrok если был ──────────────────────────────────────
_tunnel = globals().get('_tunnel')
if _tunnel is not None:
    try:
        from pyngrok import ngrok as _ng
        _ng.disconnect(_tunnel.public_url)
        _ng.kill()
    except Exception:
        pass

# ── Загружаем модели (только при первом запуске) ──────────────────────────
_analyzer = globals().get('_analyzer')
if _analyzer is None:
    import sys
    from pathlib import Path
    repo_root = Path('/work/building_analyzer')
    if not repo_root.exists():
        repo_root = Path.cwd().resolve()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    print('Загружаем модели... (1-3 мин при первом запуске)')
    from backend.ml_pipeline import FacadeAnalyzer
    _analyzer = FacadeAnalyzer()
    _analyzer.load_models()
    print(f'Модели загружены на устройство: {_analyzer.device}')
else:
    print(f'Модели уже в памяти (device={_analyzer.device}) — пропускаем загрузку')

# ── Создаём FastAPI приложение ────────────────────────────────────────────
from backend.api import create_app
_app = create_app(analyzer=_analyzer)

# ── Запускаем сервер ─────────────────────────────────────────────────────
import uvicorn
_config = uvicorn.Config(_app, host='0.0.0.0', port=8000, loop='asyncio', log_level='info')
_server = uvicorn.Server(_config)
_server_task = asyncio.ensure_future(_server.serve())

# Ждём пока сервер поднимется
import httpx
for _i in range(30):
    try:
        _r = httpx.get('http://127.0.0.1:8000/api/health', timeout=2)
        if _r.status_code == 200:
            print('Сервер запущен:', _r.json())
            break
    except httpx.RequestError:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Сервер не поднялся за 30 сек — проверьте логи выше')

# ── Подключаем ngrok ──────────────────────────────────────────────────────
_tunnel = None
if ngrok_token:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    _tunnel = ngrok.connect(8000, 'http')
    print('=' * 60)
    print('PUBLIC URL:', _tunnel.public_url)
    print('DOCS      :', _tunnel.public_url + '/docs')
    print('HEALTH    :', _tunnel.public_url + '/api/health')
    print('=' * 60)
    print('Вставьте PUBLIC URL в настройках Flutter-приложения.')
else:
    print('Сервер доступен локально: http://127.0.0.1:8000')
    print('Ngrok не подключён (нет токена)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 — STOP  (остановить сервер, модели останутся в памяти)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import asyncio

_server = globals().get('_server')
_server_task = globals().get('_server_task')
_tunnel = globals().get('_tunnel')

if _server is not None:
    _server.should_exit = True
    try:
        await asyncio.wait_for(_server_task, timeout=10)
    except asyncio.TimeoutError:
        _server_task.cancel()
    _server = None
    print('Сервер остановлен')
else:
    print('Сервер не запущен')

if _tunnel is not None:
    try:
        from pyngrok import ngrok
        ngrok.disconnect(_tunnel.public_url)
        ngrok.kill()
    except Exception as e:
        print('ngrok.disconnect:', e)
    _tunnel = None
    print('Ngrok отключён')

print('Модели остаются в памяти — можно снова запустить ячейку 2 (START)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 — SCRAPER  (опционально, разовый запуск парсера цен)        ║
# ╚══════════════════════════════════════════════════════════════════╝
from backend.scraper.worker import run_once
await run_once('all')
